# Verification in Speculative Decoding

> A 0.5B model can nearly double the speed of a 70B model while preserving exactly the Target model's output distribution. This works because Decode is serial: each Token waits for the previous one.
>
> This chapter covers the serial bottleneck, Draft-and-Verify, the $\min(1,p/q)$ acceptance rule and correction distribution, the speedup equation, and proposer families including MTP, Medusa, EAGLE, Self-Speculative, Prompt Lookup, and Lookahead.


## 1. The Serial Bottleneck of Autoregressive Decoding

```text
Target forward → Token 1
Target forward → Token 2  (waits for Token 1)
Target forward → Token 3  (waits for Token 2)
```

Quantization makes each step cheaper and KV Cache avoids recomputing history, but neither changes one confirmed Token per Target pass. Yet a forward pass can process many sequence positions in parallel, as Prefill demonstrates. Speculative decoding guesses several positions and verifies them together.


## 2. Basic Speculative-Decoding Flow

1. **Draft**: a cheap model generates $K$ candidate Tokens serially.
2. **Verify**: the Target processes all candidates in one forward pass and obtains a probability distribution at every position.
3. **Accept**: scan left to right, keep the longest accepted prefix, and correct the first rejection.

Causal masking makes parallel verification valid: position $i$ depends only on earlier candidate positions. The difficult part is choosing acceptance and correction rules that leave the final distribution exactly equal to the Target.


## 3. Acceptance and Correction

Let Draft probability be $q(x)$ and Target probability be $p(x)$. Classical speculative sampling accepts a proposed Token with probability

$$a(x)=\min\left(1,rac{p(x)}{q(x)}ight).$$

If Target likes the Token at least as much as Draft, accept it. Otherwise accept proportionally. This is probabilistic, not a Top-1 equality test.

After rejection, sampling directly from $p$ would double-count conditioned probability mass. Instead sample from

$$p'(x)=\operatorname{norm}(\max(p(x)-q(x),0)).$$

Acceptance plus this residual distribution yields exactly $p(x)$ overall, independent of proposer quality.


In [ ]:
draft_probs  = [0.80, 0.70, 0.60, 0.50]
target_probs = [0.90, 0.80, 0.20, 0.10]

for i,(q,p) in enumerate(zip(draft_probs,target_probs)):
    accept = min(1.0, p/q)
    print(f"pos {i}: target/draft={p/q:.2f}, accept_prob={accept:.2f}")


## 4. Complete Speculative Decoding

One round performs:

```text
1. Draft generates gamma candidates.
2. Target computes all candidate-position distributions in one pass.
3. Accept candidates left-to-right with probability min(1, p/q).
4. On the first rejection, sample from norm(max(p-q, 0)) and end the round.
5. If all candidates pass, sample one bonus Token from Target's next distribution.
```

The bonus is available because the verification pass already computed the following position.


In [ ]:
import numpy as np

vocab = ["to", "day", "weather", "very", "good", "cold"]
idx = {t: i for i, t in enumerate(vocab)}

# Two score tables mapping current Token to next Token: Target is sharper; Draft is a noisy imitation
rng = np.random.default_rng(0)
score_target = rng.normal(0, 2.0, (len(vocab), len(vocab)))
score_draft = score_target + rng.normal(0, 1.5, score_target.shape)

def dist(scores, token):
    """Turn the score row for the current Token into a next-Token probability distribution."""
    z = scores[idx[token]]
    e = np.exp(z - z.max())
    return e / e.sum()

def sample_from(p):
    return vocab[int(np.random.choice(len(p), p=p))]

def speculative_step(context_token, gamma=4):
    """Run one speculative-decoding round and return all confirmed Tokens, using one target forward pass."""
    draft_tokens = []
    cur = context_token
    for _ in range(gamma):
        nxt = sample_from(dist(score_draft, cur))
        draft_tokens.append(nxt)
        cur = nxt

    accepted = []
    cur = context_token
    for t in draft_tokens:
        p = dist(score_target, cur)   # target view at this position
        q = dist(score_draft, cur)    # evidence used by the draft
        if np.random.random() < min(1.0, p[idx[t]] / q[idx[t]]):
            accepted.append(t)
            cur = t
        else:
            residual = np.maximum(p - q, 0)   # correction distribution: probability mass present in target but not draft
            accepted.append(sample_from(residual / residual.sum()))
            break
    if len(accepted) == len(draft_tokens):
        accepted.append(sample_from(dist(score_target, cur)))   # bonus Token
    return accepted

np.random.seed(42)
print("One sample round:", speculative_step("to", gamma=4))
print("Another round:", speculative_step("to", gamma=4))


In [ ]:
# Run 2,000 rounds and count Tokens confirmed per target forward pass
np.random.seed(42)
rounds = 2000
accepted_counts = np.array([len(speculative_step("to", gamma=4)) for _ in range(rounds)])

print('Number of tokens: ', round(accepted_counts.mean(), 3))
print('"distribution:"', {k: int((accepted_counts == k).sum()) for k in range(1, 6)})
print("Notice that 4 is absent: when all four draft Tokens are accepted, a bonus Token makes the count 5")
print()
print("Key observation: ordinary Decode confirms only one token per forward pass;")
print("One target forward plus four cheap draft steps confirms an average of",
      round(accepted_counts.mean(), 2), "Tokens")


In [ ]:
# Accepted-length distribution: most rounds confirm 1-3 Tokens; accepting all five is uncommon
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.2))
plt.hist(accepted_counts, bins=np.arange(0.5, 6.5, 1),
         edgecolor="black", color="tab:blue")
plt.xticks(range(1, 6))
plt.xlabel("tokens confirmed per target forward")
plt.ylabel("rounds")
plt.title("Acceptance length distribution (gamma = 4)")
plt.show()


## 5. Speedup Analysis

An approximate speedup is

$$	ext{speedup}pproxrac{	ext{average confirmed Tokens per round}}
{1+\gamma	imes	ext{Draft cost per step in Target units}}.$$

Higher acceptance helps, but larger $\gamma$ can waste rejected Draft work. A Draft that is too small is cheap but inaccurate; one that is too large consumes the savings. Real systems tune proposer size, candidate length, and workload—repetitive tasks generally accept more than open-ended generation.


In [ ]:
def toy_speedup(k, accept_rate, draft_cost_ratio):
    # Teaching proxy: each round yields 1 plus the number of accepted draft Tokens in expectation
    expected_tokens = 1 + k * accept_rate
    cost = 1 + k * draft_cost_ratio
    return expected_tokens / cost

for a in [0.3,0.6,0.9]:
    print("accept", a, "proxy speedup", round(toy_speedup(4,a,0.08),2))


## 6. Distribution-Consistency Experiment

We verify the claim with Monte Carlo sampling. Target is `[0.7, 0.3]` and Draft deliberately differs at `[0.5, 0.5]`. After 50,000 speculative samples, empirical frequencies should converge to Target, showing that correction removes Draft bias.


In [ ]:
import random, collections

p = [0.7, 0.3]  # target
q = [0.5, 0.5]  # draft

def sample(dist):
    r = random.random()
    return 0 if r < dist[0] else 1

def speculative_one():
    x = sample(q)
    if random.random() < min(1.0, p[x] / q[x]):
        return x
    residual = [max(p[i]-q[i],0.0) for i in range(2)]
    s = sum(residual)
    if s == 0:
        return sample(p)
    residual = [v/s for v in residual]
    return sample(residual)

random.seed(42)
n=50000
c=collections.Counter(speculative_one() for _ in range(n))
print("target:", p)
print("speculative empirical:", [round(c[i]/n,3) for i in range(2)])


The experiment is speculative decoding's identity check: regardless of Draft preference, final frequencies match Target. Speed is an optimization; unchanged distribution is the correctness requirement.


## 7. Proposer Families: Different Ways to Guess

The original method uses a separate small model, which requires training, memory, and tokenizer alignment. Later variants retain guess → verify → accept but replace the proposer.

| Family | Proposer | Examples | Training | Memory |
|:---|:---|:---|:---|:---|
| Separate model | 0.1B–1B model | classical speculative decoding | yes | another model |
| Target-derived | added heads/layers or shallow Target | MTP, Medusa, EAGLE, LayerSkip | usually | small or zero |
| Non-learned | lookup or iteration | Prompt Lookup, Lookahead | no | near zero |

For every method ask: who proposes, who verifies, and how many Tokens one Target pass confirms.


### 7.1 MTP: Learn to Predict Multiple Future Tokens

**Multi-Token Prediction (MTP)** trains a model to predict not only $t+1$ but farther Tokens. A small MTP module receives the Target hidden state and the next-Token Embedding, then predicts another future Token. It therefore proposes from richer information than an independent Draft.

The auxiliary loss shares data with the main objective. At inference the MTP module runs cheaply for a few steps, and standard rejection sampling verifies its proposals. Benefits are one tokenizer, no second full model, and higher acceptance; the cost is planning or post-training the extra module.


In [ ]:
import numpy as np

vocab = ["to", "day", "weather", "very", "good", "cold"]
V = len(vocab)
rng = np.random.default_rng(7)

# Target conditions on the previous two Tokens, so it has more information than an independent draft
score_target = rng.normal(0, 2.0, (V, V, V))   # (prev2, prev1, next)

# Independent small model: sees one previous Token plus noise, simulating no access to hidden states
score_indep = score_target.mean(axis=0) + rng.normal(0, 1.5, (V, V))

# MTP-style draft: sees the same information as target plus slight noise, simulating access to hidden states
score_mtp = score_target + rng.normal(0, 0.6, score_target.shape)

def dist3(scores, t2, t1):
    z = scores[vocab.index(t2), vocab.index(t1)]
    e = np.exp(z - z.max())
    return e / e.sum()

def dist2(scores, t1):
    z = scores[vocab.index(t1)]
    e = np.exp(z - z.max())
    return e / e.sum()

def sample_idx(p):
    return int(np.random.choice(V, p=p))

def one_round(draft_kind, gamma=4):
    'Number of tokens: '
    t2, t1 = "to", "day"   # fixed context for comparison
    draft, n_ok = [], 0
    c2, c1 = t2, t1
    for _ in range(gamma):
        if draft_kind == "indep":
            nxt = vocab[sample_idx(dist2(score_indep, c1))]
        else:
            nxt = vocab[sample_idx(dist3(score_mtp, c2, c1))]
        draft.append(nxt)
        c2, c1 = c1, nxt
    # A target forward is serially equivalent to verifying each position with the true conditional distribution
    c2, c1 = t2, t1
    for tok in draft:
        p = dist3(score_target, c2, c1)
        q = dist3(score_mtp, c2, c1) if draft_kind == "mtp" else dist2(score_indep, c1)
        i = vocab.index(tok)
        if np.random.random() < min(1.0, p[i] / max(q[i], 1e-9)):
            n_ok += 1
            c2, c1 = c1, tok
        else:
            break
    return n_ok + 1   # +1 bonus Token

np.random.seed(42)
rounds = 2000
indep_lens = np.array([one_round("indep") for _ in range(rounds)])
mtp_lens = np.array([one_round("mtp") for _ in range(rounds)])

print("Independent small draft: confirms", round(indep_lens.mean(), 2), "Tokens per round on average")
print("MTP-style draft:         confirms", round(mtp_lens.mean(), 2), "Tokens per round on average")
print()
print("Key observation: the closer the draft information is to the target model's information, the higher the acceptance rate;")
print("MTP's advantage is not faster guessing; it guesses from the target's hidden state.")


In [ ]:
# Compare accepted-length distributions for the two drafts
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.2))
bins = np.arange(0.5, 6.5, 1)
plt.hist(indep_lens, bins=bins, alpha=0.6, label="independent draft", edgecolor="black")
plt.hist(mtp_lens, bins=bins, alpha=0.6, label="MTP-style draft", edgecolor="black")
plt.xticks(range(1, 6))
plt.xlabel("tokens confirmed per target forward")
plt.ylabel("rounds")
plt.title("MTP-style draft sees more, accepts more")
plt.legend()
plt.show()


### 7.1.1 Adding MTP to a Model

Three routes are available:

1. **Pretrain with MTP** using $\mathcal L=\mathcal L_{main}+\lambda\mathcal L_{MTP}$ so base representations and the module co-adapt.
2. **Post-train an MTP module** while freezing an existing Target. This is cheaper and cannot damage base weights, but may have a lower ceiling.
3. **Use multi-future-token targets without a separate module**, accepting lower proposal quality for zero added parameters.

Training uses the true $t+1$ through teacher forcing, while inference feeds the module its own guessed $t+1$. This exposure gap compounds with depth, which is why MTP modules are usually shallow.


### 7.1.2 Enabling MTP in an Inference System

Serving frameworks expose MTP as configuration when the checkpoint contains compatible weights:

```bash
vllm serve deepseek-ai/DeepSeek-V3 \
  --speculative-config '{"method":"deepseek_mtp","num_speculative_tokens":3}'
```

`num_speculative_tokens` controls proposal length. Reported speedups are below the theoretical four Tokens per round because conditional acceptance falls at later positions and the MTP module itself costs compute.


In [ ]:
# Verify the calculation from Section 5: with 85% acceptance and three guesses, how many Tokens does one round confirm?
def expected_tokens(a, gamma):
    total, keep = 1.0, 1.0   # 1.0 is the bonus Token
    for _ in range(gamma):
        keep *= a
        total += keep
    return total

for a in [0.75, 0.85, 0.95]:
    n = expected_tokens(a, 3)
    print(f"Acceptance {a:.2f}: one round confirms {n:.2f} Tokens, "
          f"for an ideal speedup around {n/1.2:.1f}x")

print()
print("Key observation: lowering acceptance from 0.95 to 0.75 cuts speedup from 3.1x to 2.3x;")
print("This is an idealized account; real systems also pay verification-memory and scheduling overhead.")
print("DeepSeek-V3's reported 1.8x is the resulting end-to-end number.")


### 7.1.3 Why MTP Preserves Quality

Distribution preservation depends on the verifier, not proposer identity. MTP still uses $\min(1,p/q)$ acceptance and residual correction, so it changes acceptance rate but not final output probabilities.

Base capability is also preserved because the module is a bypass. It may be disabled to recover ordinary Target behavior, and post-training can freeze the Target entirely. To evaluate any proposal method's theoretical quality, inspect whether verification still uses exact rejection sampling.


### 7.1.4 When Speculation Should Be Disabled

At small batch sizes, Decode is often memory-bandwidth-bound: verifying several positions reuses one weight load and exploits idle compute. At large batches, compute is already saturated; verifying four positions approaches four times the FLOPs, and insufficient acceptance can reduce throughput.

Serving systems therefore often enable speculation dynamically for low concurrency and disable it under high load.


In [ ]:
# Compare speedups under two resource bottlenecks
import numpy as np
import matplotlib.pyplot as plt

accept = 0.85          # per-step acceptance rate
gammas = np.arange(1, 7)
tokens = [expected_tokens(accept, g) for g in gammas]

def verify_cost(g, regime):
    """Cost of one verification pass in units of a normal Decode step."""
    if regime == "memory-bound":
        return 1.0 + 0.05 * g      # same weight movement, with a little extra computation
    else:
        return 1.0 + g             # compute-bound: FLOPs grow linearly with verified positions

plt.figure(figsize=(6, 3.2))
plt.plot(gammas, [t / verify_cost(g, "memory-bound") for t, g in zip(tokens, gammas)],
         "o-", label="memory-bound (small batch)")
plt.plot(gammas, [t / verify_cost(g, "compute-bound") for t, g in zip(tokens, gammas)],
         "s-", label="compute-bound (large batch)")
plt.axhline(1.0, linestyle="--", color="gray", label="no speculation")
plt.xlabel("gamma (tokens drafted per round)")
plt.ylabel("speedup")
plt.title("Same acceptance, opposite verdict at different batch sizes")
plt.legend()
plt.show()

print("Key observation: for the same model and acceptance rate, a larger gamma helps more at small batch sizes,")
print("At large batch, the 1/(1+gamma) factor erodes speedup; maximizing gamma can approach break-even.")


### 7.1.5 Hands-On: Add MTP to MiniGPT

We add a small MTP head to the earlier MiniGPT. The main head predicts $t+1$; the MTP head receives $[h_t,\operatorname{emb}(t+1)]$ and predicts $t+2$. Both losses are trained together on a small patterned corpus, matching the structural idea of DeepSeek-V3 at educational scale.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

corpus = ("the cat sat on the mat . the dog ran in the park . "
          "the cat ran in the park . the dog sat on the mat . "
          "a bird flew over the park . a fish swam in the pond . ")
text = corpus * 40
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
V, D, S = len(chars), 64, 32          # vocabulary / model width / context length
print("vocab:", V, "corpus tokens:", len(data))

def pos_enc(seq_len, d):
    """The same sinusoidal position encoding as Part 1."""
    pos = torch.arange(seq_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d, 2).float() * (-np.log(10000.0) / d))
    pe = torch.zeros(seq_len, d)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

class Block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, 4, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x):
        mask = torch.triu(torch.ones(x.size(1), x.size(1)) * float("-inf"), 1)
        h, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.ln1(x + h)
        return self.ln2(x + self.ff(x))

class MiniGPT_MTP(nn.Module):
    """First LLM plus MTP head: main head predicts t+1; MLP([h_t; emb(t+1)]) predicts t+2."""
    def __init__(self, v, d):
        super().__init__()
        self.emb = nn.Embedding(v, d)
        self.blocks = nn.ModuleList([Block(d) for _ in range(2)])
        self.head = nn.Linear(d, v)                # main head: predict t+1
        self.mtp = nn.Sequential(                  # MTP head: predict t+2
            nn.Linear(2 * d, 2 * d), nn.GELU(), nn.Linear(2 * d, v))

    def hidden(self, idx):
        x = self.emb(idx) + pos_enc(idx.size(1), D)
        for b in self.blocks:
            x = b(x)
        return x

    def forward(self, idx):
        h = self.hidden(idx)
        return self.head(h), h

model = MiniGPT_MTP(V, D)
print("Parameter count:", sum(p.numel() for p in model.parameters()))


The training loop uses one batch and one forward pass. The main objective predicts $t+1$, while the MTP objective teacher-forces the true $t+1$ to predict $t+2$, weighted by $\lambda=0.3$.


In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)

def get_batch():
    i = torch.randint(0, len(data) - S - 2, (32,))
    x = torch.stack([data[j:j + S] for j in i])
    y1 = torch.stack([data[j + 1:j + S + 1] for j in i])   # t+1 target for main head
    y2 = torch.stack([data[j + 2:j + S + 2] for j in i])   # t+2 target for MTP head
    return x, y1, y2

for step in range(400):
    x, y1, y2 = get_batch()
    logits, h = model(x)
    e_next = model.emb(y1)                     # teacher forcing: supply the true t+1
    mtp_logits = model.mtp(torch.cat([h, e_next], dim=-1))
    loss = F.cross_entropy(logits.reshape(-1, V), y1.reshape(-1)) \
         + 0.3 * F.cross_entropy(mtp_logits.reshape(-1, V), y2.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 100 == 0 or step == 399:
        print(f"step {step:3d}  loss {loss.item():.3f}")

print()
print("Key observation: both losses decrease, so the MTP head does not disrupt the main task;")
print("Instead, it forces the hidden state to encode information one additional step ahead.")


For evaluation, the main head proposes the first Token and the MTP head proposes the next two using its own previous guess, exposing the train/inference gap. Target then verifies once with exact acceptance and correction. A global unigram proposer provides a baseline under the identical verifier.


In [ ]:
def mtp_draft(model, x, gamma=3):
    """Guess gamma Tokens serially and return a list of (Token, draft probability q)."""
    picks = []
    with torch.no_grad():
        logits, h = model(x)
        q = F.softmax(logits[0, -1], dim=-1)
        nxt = int(torch.multinomial(q, 1))     # first Token: sampled by main head and always accepted
        picks.append((nxt, float(q[nxt])))
        h_last, t_last = h[0, -1], nxt
        for _ in range(gamma - 1):
            mlp_in = torch.cat([h_last, model.emb(torch.tensor([t_last]))[0]])
            q = F.softmax(model.mtp(mlp_in), dim=-1)
            nxt = int(torch.multinomial(q, 1))
            picks.append((nxt, float(q[nxt])))
            t_last = nxt                       # relay: consume the previous self-generated output
    return picks

def speculative_round(model, x, gamma=3, use_mtp=True):
    """One full speculative round: draft guesses, target verifies once, then accept with min(1, p/q)."""
    if use_mtp:
        picks = mtp_draft(model, x, gamma)
    else:                                      # baseline: blind global-unigram guesses
        uni = torch.bincount(data, minlength=V).float()
        uni = uni / uni.sum()
        picks = [(int(torch.multinomial(uni, 1)), float(uni[0])) for _ in range(gamma)]
        picks = [(t, float(uni[t])) for t, _ in picks]
    toks = [t for t, _ in picks]
    ext = torch.tensor(toks[:-1], dtype=torch.long).unsqueeze(0)
    with torch.no_grad():
        logits, _ = model(torch.cat([x, ext], dim=1))
    p_all = F.softmax(logits[0], dim=-1)
    n_ok = 0
    for k, (tok, q) in enumerate(picks):
        pos = x.size(1) + k - 1                # position that predicts picks[k]
        p = float(p_all[pos, tok])
        if torch.rand(1).item() < min(1.0, p / max(q, 1e-9)):
            n_ok += 1
        else:
            break
    return n_ok + 1                            # +1 bonus Token

torch.manual_seed(0)
starts = torch.randint(0, len(data) - S - 2, (300,))
mtp_lens = np.array([speculative_round(model, data[j:j + S].unsqueeze(0), use_mtp=True)
                     for j in starts])
uni_lens = np.array([speculative_round(model, data[j:j + S].unsqueeze(0), use_mtp=False)
                     for j in starts])

print("MTP draft:       confirms", round(mtp_lens.mean(), 2), "Tokens per round on average")
print("Unigram baseline: confirms", round(uni_lens.mean(), 2), "Tokens per round on average")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3.2))
bins = np.arange(0.5, 5.5, 1)
plt.hist(uni_lens, bins=bins, alpha=0.6, label="unigram draft (no info)", edgecolor="black")
plt.hist(mtp_lens, bins=bins, alpha=0.6, label="MTP draft (sees hidden state)", edgecolor="black")
plt.xticks(range(1, 5))
plt.xlabel("tokens confirmed per target forward")
plt.ylabel("rounds")
title = f"MTP head on our first LLM: {mtp_lens.mean():.1f} vs {uni_lens.mean():.1f}"
plt.title(title + " tokens per round")
plt.legend()
plt.show()

With the same verifier and one Target pass, hidden-state-informed MTP confirms far more Tokens than a unigram guesser. `self.mtp` corresponds to the MTP module, teacher forcing to the training input, and feeding each guess back to the module to inference-time error accumulation. Vary $\gamma$ or module depth to observe the acceptance trade-off.


### 7.2 Medusa: Multiple Prediction Heads

Medusa attaches $K$ small heads to the Target's final hidden state, where head $k$ predicts Token $t+k+1$. Farther heads do not observe intervening sampled Tokens, so their accuracy decays. Rather than forming one chain, top candidates form a tree that Target verifies in one tree-Attention pass.

Original Medusa commonly uses “typical acceptance,” accepting Tokens in a high-probability Target region rather than exact $p/q$ rejection sampling. This improves throughput but no longer guarantees an identical distribution. Medusa proposes with added heads and verifies with Target plus a tree mask.


### 7.3 EAGLE: Predict Features, Not Tokens

EAGLE trains a lightweight Draft layer in the Target's penultimate feature space. It predicts the next hidden feature, then uses Target's original `lm_head` to decode a Token. Continuous high-dimensional features retain more context than discrete Tokens and are empirically easier to extrapolate.

The Draft recursively combines a feature with the guessed Token Embedding, builds candidate trees, and uses exact rejection sampling. EAGLE-2 grows trees dynamically from confidence; EAGLE-3 improves feature alignment across layers. It needs one or two trained Draft layers but preserves distribution.


### 7.4 Self-Speculative Decoding and LayerSkip

Self-Speculative Decoding uses the Target's own shallow subnetwork to propose and the full network to verify. For example, a 32-layer model may exit after layer 16 during Draft and run all layers for Verify. It adds no model memory.

Ordinary shallow states may poorly match the final distribution. LayerSkip trains with random late-layer dropping and early-exit losses so intermediate layers become useful predictors. This route minimizes engineering state but benefits from training designed for early exit.


### 7.5 Prompt Lookup: Non-Learned Proposals

Prompt Lookup finds an n-gram matching the latest generated Tokens inside the Prompt or prior output and proposes the following Tokens. Summarization, RAG quotation, code completion, and entity repetition often copy existing spans, making lookup surprisingly effective.

It requires no parameters, memory, or training: a sliding-window hash table proposes matches and Target verifies them. Open-ended creative tasks rarely match and should disable it automatically.


In [ ]:
import random

prompt = "France is Paris . Japan is Tokyo .".split()

def target_next(generated, task):
    """Toy target: copy tasks repeat prompt patterns; creative tasks behave randomly."""
    if task == "copy":
        if generated[-1] == "is":
            return "Paris" if random.random() < 0.75 else "Tokyo"
        if generated[-1] in ("Paris", "Tokyo"):
            return "." if random.random() < 0.9 else random.choice(["Paris", "Tokyo"])
        return "is"
    return random.choice(["Berlin", "Cairo", "Lima", "Rome", "Oslo"])

def prompt_lookup(generated, n=2, k=4):
    """Find the last n Tokens as an n-gram in the prompt and copy the following k Tokens as candidates."""
    key = tuple(generated[-n:])
    for i in range(len(prompt) - n):
        if tuple(prompt[i:i+n]) == key:
            return prompt[i+n:i+n+k]
    return []

def run(task, rounds=3000):
    accepted = []
    for _ in range(rounds):
        gen = ["The", "capital", "of", "France", "is"]
        draft = prompt_lookup(gen)
        n_ok = 0
        for tok in draft:
            if target_next(gen, task) == tok:
                n_ok += 1
                gen.append(tok)
            else:
                break
        accepted.append(n_ok + 1)   # +1 bonus
    return sum(accepted) / len(accepted)

random.seed(42)
print("Copy task:     confirms", round(run("copy"), 2), "Tokens per round on average")
print("Creative task: confirms", round(run("creative"), 2), "Tokens per round on average")
print()
print("Key observation: the same lookup logic can have very different value when the task changes;")
print("Speculative decoding has no universal solution; choose the proposer for the workload.")


In [ ]:
# Compare prompt-lookup gains under the two tasks
import matplotlib.pyplot as plt

random.seed(42)
copy_avg = run("copy")
creative_avg = run("creative")

plt.figure(figsize=(5, 3))
plt.bar(["copy-style task", "creative task"], [copy_avg, creative_avg],
        color=["tab:green", "tab:red"], edgecolor="black")
plt.axhline(1.0, linestyle="--", color="gray", label="no speculation (1.0)")
plt.ylabel("tokens per target forward")
plt.title("Prompt lookup: free lunch only for copy-like tasks")
plt.legend()
plt.show()


### 7.6 Lookahead: Generation as an Iterative System

Lookahead Decoding initializes several future positions and updates them in parallel with Jacobi-style iterations. Each Target pass revises all positions; intermediate guesses create an n-gram pool whose candidates are verified in later rounds.

It needs no separate model or head and retains exact verification. Its speedups may trail learned feature proposers, but it obtains parallelism solely from the mathematical structure of autoregressive constraints.


### 7.7 Summary of Proposer Families

| Method | Proposer | Training | Extra Memory | Exact Distribution | Key Trait |
|:---|:---|:---|:---|:---|:---|
| Classical | separate small model | yes | small model | yes | simplest concept, two models |
| MTP | hidden state + module | yes | small | yes | high-information proposal |
| Medusa | extra heads | head training | small | not with typical acceptance | tree verification |
| EAGLE | feature Draft layer | yes | 1–2 layers | yes | predicts features |
| LayerSkip | Target shallow layers | training helps | zero | yes | self-proposal |
| Prompt Lookup | n-gram table | no | zero | yes | excellent for copying |
| Lookahead | Jacobi + n-gram pool | no | zero | yes | schedule-only parallelism |

Better access to Target information usually raises acceptance. Training, memory, and workload determine which proposer is appropriate. Always identify proposer, verifier, and confirmed Tokens per Target pass.


- Autoregressive Decode is serial even when compute is available.
- Draft proposes $\gamma$ Tokens and Target verifies them in one pass.
- Acceptance is probabilistic: $\min(1,p/q)$, not Top-1 equality.
- Residual correction `norm(max(p-q, 0))` preserves Target distribution.
- Speedup depends on acceptance, proposal length, and Draft cost.
- MTP uses hidden states, EAGLE predicts features, Medusa adds heads, LayerSkip uses shallow layers, Prompt Lookup copies, and Lookahead iterates; the verification skeleton remains.


## Exercises

**Exercise 1: Acceptance Probability Calculation**

The Draft model assigns token A a probability of 0.5, and the Target model assigns it 0.8. What is the acceptance probability?

Hint: min(1, p_target / p_draft)

### Exercise 1: Implement the Correction Distribution

After Draft rejection, sample only from normalized `max(p-q, 0)`.

Hint: apply `np.maximum(p - q, 0)`, then divide by its sum.


In [ ]:
# Exercise 1: fill in the correction distribution

import numpy as np

p = np.array([0.5, 0.3, 0.2])   # target
q = np.array([0.6, 0.2, 0.2])   # draft

def correction_dist(p, q):
    """Return the correction distribution sampled after rejection."""
    residual = np.zeros_like(p)
    # TODO: Replace the triple-quoted content below with your code
    """Set residual to max(p - q, 0), then normalize."""
    return residual

corr = correction_dist(p, q)
assert abs(corr.sum() - 1.0) < 1e-9, corr
assert corr[0] == 0.0, corr
print("Exercise 1 passed: the correction distribution retains only probability mass where target exceeds draft")


### Exercise 2: Expected Accepted Tokens

For conditional acceptance probabilities $a_1,a_2,a_3$, the expected number of accepted Draft Tokens is $a_1+a_1a_2+a_1a_2a_3$.

Hint: maintain a cumulative product and add it at each position.


In [ ]:
# Exercise 2: fill in expected accepted length

def expected_accepted(accept_probs):
    'Number of tokens: '
    # TODO: Replace the triple-quoted content below with your code
    """Multiply and accumulate a1 + a1*a2 + a1*a2*a3 + ..."""

assert abs(expected_accepted([0.9, 0.8, 0.7]) - 2.124) < 1e-9
assert expected_accepted([1.0, 1.0, 1.0]) == 3.0
print("Exercise 2 passed: you can quantify how many tokens one round confirms")


### Exercise 3: Is Speculation Worth It?

One round costs one Target pass plus $\gamma$ Draft passes. With relative Draft cost `draft_cost` and average confirmed Tokens `avg_tokens`, compute `avg_tokens / (1 + gamma * draft_cost)`.

Hint: compare the result with 1.0.


In [ ]:
# Exercise 3: fill in speedup

def speedup(avg_tokens, gamma, draft_cost):
    """Return speedup relative to ordinary Decode."""
    # TODO: Replace the triple-quoted content below with your code
    """Round cost = 1 + gamma * draft_cost in target-forward equivalents."""

assert speedup(3.0, 4, 0.1) > 1.5
assert speedup(1.5, 8, 0.2) < 1.0
print("Exercise 3 passed: speedup is not free; acceptance and draft cost must be considered together")
